# 04. LightGBM 하이퍼파라미터 튜닝

`03_cv_and_prior_correction.ipynb`에서 결정한 것들을 그대로 가져옵니다.
- 고정 CV 폴드 (`processed/cv_folds.csv`)
- 결측 복구 피처 (`physical_activity_level_recovered`, `sleep_duration_recovered`)
- 최종 결정규칙: **사전확률 보정** (`argmax P(c|x)/P(c)`), class_weight와 동시 사용 금지
- 03의 기본 파라미터 CV 결과: **OOF balanced accuracy 0.94980** (튜닝 전 baseline)

여기서는 Optuna로 LightGBM 하이퍼파라미터를 탐색해서 이 baseline을 얼마나 개선할 수 있는지 확인합니다.

> 가이드 문서(팀원 분석) 기준으로 이 대회의 이론적 상한은 balanced accuracy ≈ 0.941~0.951 근처이고, 03에서 이미 0.9498을 달성했기 때문에 **튜닝으로 얻을 수 있는 추가 개선폭은 크지 않을 것으로 예상**합니다(+0.001~0.003 수준). 그래도 재현 가능한 방식으로 탐색해봅니다.

커널: **Python (teammate)** (lightgbm, optuna 설치된 환경)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
TARGET = "health_condition"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
folds = pd.read_csv(OUT_DIR / "cv_folds.csv")
train = train.merge(folds, on="id", how="left")
assert train["fold"].isna().sum() == 0

print(train.shape, test.shape)

/opt/anaconda3/envs/teammate/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(690088, 16) (295753, 14)


## 1. 피처 엔지니어링 (03과 동일)

In [2]:
def add_recovery_features(df, step_tertiles, sleep_quality_cond):
    df = df.copy()
    activity_proxy = pd.cut(
        df["step_count"], bins=[-np.inf, step_tertiles[0], step_tertiles[1], np.inf],
        labels=["sedentary", "moderate", "active"],
    ).astype(object)
    df["physical_activity_level_recovered"] = df["physical_activity_level"]
    missing_activity = df["physical_activity_level"].isna()
    df.loc[missing_activity, "physical_activity_level_recovered"] = activity_proxy[missing_activity]
    df["physical_activity_level_recovered"] = df["physical_activity_level_recovered"].fillna("moderate")

    df["sleep_duration_recovered"] = df["sleep_duration"]
    missing_sleep = df["sleep_duration"].isna()
    fallback_median = sleep_quality_cond.get("missing", sleep_quality_cond["average"])
    mapped = df.loc[missing_sleep, "sleep_quality"].map(sleep_quality_cond).fillna(fallback_median)
    df.loc[missing_sleep, "sleep_duration_recovered"] = mapped

    df["stress_level_isnull"] = df["stress_level"].isna().astype(np.int8)
    df["sleep_duration_isnull"] = df["sleep_duration"].isna().astype(np.int8)
    df["physical_activity_level_isnull"] = df["physical_activity_level"].isna().astype(np.int8)
    return df


step_tertiles = train["step_count"].quantile([1/3, 2/3]).values
sleep_quality_cond = train.groupby("sleep_quality")["sleep_duration"].median().to_dict()
sleep_quality_cond["missing"] = train["sleep_duration"].median()

train = add_recovery_features(train, step_tertiles, sleep_quality_cond)
test = add_recovery_features(test, step_tertiles, sleep_quality_cond)

NUMERIC_COLS = [
    "sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
NOMINAL_COLS = ["diet_type", "gender"]
FLAG_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

numeric_medians = train[NUMERIC_COLS].median()
for df in (train, test):
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(numeric_medians[col])

categorical_cols = list(ORDINAL_COLS.keys()) + NOMINAL_COLS
for df in (train, test):
    for col in categorical_cols:
        df[col] = df[col].fillna("missing")

for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    encoder = OrdinalEncoder(categories=[categories])
    train[col] = encoder.fit_transform(train[[col]])
    test[col] = encoder.transform(test[[col]])

train_ohe = pd.get_dummies(train[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = pd.get_dummies(test[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)
train = pd.concat([train.drop(columns=NOMINAL_COLS), train_ohe], axis=1)
test = pd.concat([test.drop(columns=NOMINAL_COLS), test_ohe], axis=1)

FEATURE_COLS = NUMERIC_COLS + list(ORDINAL_COLS.keys()) + FLAG_COLS + list(train_ohe.columns)

target_encoder = LabelEncoder()
train["target_enc"] = target_encoder.fit_transform(train[TARGET])
lgb_class_order = list(target_encoder.classes_)
train_priors = train[TARGET].value_counts(normalize=True).to_dict()

print(len(FEATURE_COLS), "features")

22 features


In [3]:
def prior_corrected_predict(proba, class_order, priors):
    prior_arr = np.array([priors[c] for c in class_order])
    scores = proba / prior_arr
    return np.array(class_order)[scores.argmax(axis=1)]


def cv_balanced_accuracy(params, train, feature_cols, n_folds=N_FOLDS):
    """고정 폴드로 학습 -> OOF 확률 -> 사전확률 보정 -> balanced accuracy 반환"""
    oof_proba = np.zeros((len(train), 3))
    for fold in range(n_folds):
        tr_idx = train["fold"] != fold
        va_idx = train["fold"] == fold
        X_tr, y_tr = train.loc[tr_idx, feature_cols], train.loc[tr_idx, "target_enc"]
        X_va, y_va = train.loc[va_idx, feature_cols], train.loc[va_idx, "target_enc"]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr, eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        oof_proba[va_idx.values] = model.predict_proba(X_va)

    pred = prior_corrected_predict(oof_proba, lgb_class_order, train_priors)
    ba = balanced_accuracy_score(train[TARGET].values, pred)
    return ba, oof_proba

## 2. baseline 재확인 (03과 동일 파라미터)

In [4]:
baseline_params = dict(
    objective="multiclass", num_class=3, n_estimators=500,
    learning_rate=0.05, num_leaves=63, subsample=0.8,
    colsample_bytree=0.8, random_state=SEED, verbosity=-1,
)
baseline_ba, _ = cv_balanced_accuracy(baseline_params, train, FEATURE_COLS)
print(f"baseline OOF balanced accuracy: {baseline_ba:.5f}  (03 결과 0.94980과 비교)")

baseline OOF balanced accuracy: 0.94980  (03 결과 0.94980과 비교)


## 3. Optuna 탐색

속도를 위해 폴드 1개(fold 0)로만 빠르게 스코어를 매겨 탐색하고, 최종 후보만 5-fold 전체로 재검증합니다.

In [5]:
def quick_score(params, fold=0):
    tr_idx = train["fold"] != fold
    va_idx = train["fold"] == fold
    X_tr, y_tr = train.loc[tr_idx, FEATURE_COLS], train.loc[tr_idx, "target_enc"]
    X_va, y_va = train.loc[va_idx, FEATURE_COLS], train.loc[va_idx, "target_enc"]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    proba = model.predict_proba(X_va)
    pred = prior_corrected_predict(proba, lgb_class_order, train_priors)
    return balanced_accuracy_score(train.loc[va_idx, TARGET].values, pred)


def objective(trial):
    params = dict(
        objective="multiclass", num_class=3, random_state=SEED, verbosity=-1,
        n_estimators=1000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 200),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )
    return quick_score(params)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("best fold0 BA:", study.best_value)
print("best params:", study.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.950189:   0%|          | 0/30 [01:00<?, ?it/s]

Best trial: 0. Best value: 0.950189:   3%|▎         | 1/30 [01:00<29:02, 60.08s/it]

Best trial: 0. Best value: 0.950189:   3%|▎         | 1/30 [01:14<29:02, 60.08s/it]

Best trial: 0. Best value: 0.950189:   7%|▋         | 2/30 [01:14<15:27, 33.13s/it]

...

Best trial: 15. Best value: 0.950796:  97%|█████████▋| 29/30 [14:05<00:20, 20.31s/it]

Best trial: 15. Best value: 0.950796:  97%|█████████▋| 29/30 [14:31<00:20, 20.31s/it]

Best trial: 15. Best value: 0.950796: 100%|██████████| 30/30 [14:31<00:00, 21.95s/it]

Best trial: 15. Best value: 0.950796: 100%|██████████| 30/30 [14:31<00:00, 29.06s/it]

best fold0 BA: 0.9507960076720944
best params: {'learning_rate': 0.047792122422826176, 'num_leaves': 19, 'max_depth': 9, 'min_child_samples': 172, 'subsample': 0.8929201812783885, 'colsample_bytree': 0.7318659835628288, 'reg_alpha': 0.0005023614837892232, 'reg_lambda': 0.32275087452118445}


## 4. 최적 파라미터를 5-fold 전체로 재검증

In [6]:
best_params = dict(
    objective="multiclass", num_class=3, random_state=SEED, verbosity=-1,
    n_estimators=1000, **study.best_params,
)

tuned_ba, tuned_oof_proba = cv_balanced_accuracy(best_params, train, FEATURE_COLS)

print(f"baseline (03)      : 0.94980")
print(f"baseline (재확인)   : {baseline_ba:.5f}")
print(f"tuned (5-fold 전체) : {tuned_ba:.5f}")
print(f"개선폭              : {tuned_ba - baseline_ba:+.5f}")

baseline (03)      : 0.94980
baseline (재확인)   : 0.94980
tuned (5-fold 전체) : 0.94987
개선폭              : +0.00007


## 5. 최종 모델 학습 + test 예측 + 제출 파일 생성

In [7]:
# baseline과 tuned 중 CV 기준으로 더 나은 쪽을 최종 채택
final_params = best_params if tuned_ba > baseline_ba else baseline_params
final_ba = max(tuned_ba, baseline_ba)
print("최종 채택:", "tuned" if final_params is best_params else "baseline (03과 동일)")
print("채택된 CV balanced accuracy:", round(final_ba, 5))

# n_estimators는 early stopping 없이 전체 학습에 쓸 것이므로 고정값으로 재설정
final_params_full = dict(final_params)
final_params_full["n_estimators"] = 500

final_model = lgb.LGBMClassifier(**final_params_full)
final_model.fit(train[FEATURE_COLS], train["target_enc"])

test_proba = final_model.predict_proba(test[FEATURE_COLS])
test_pred = prior_corrected_predict(test_proba, lgb_class_order, train_priors)

submission = pd.DataFrame({"id": test["id"], TARGET: test_pred})
submission.to_csv(OUT_DIR / "submission_v2_tuned.csv", index=False)
print("saved:", OUT_DIR / "submission_v2_tuned.csv")
submission[TARGET].value_counts(normalize=True)

최종 채택: tuned
채택된 CV balanced accuracy: 0.94987


saved: ../playground-series-s6e7/processed/submission_v2_tuned.csv


health_condition
at-risk      0.810061
unhealthy    0.115884
fit          0.074055
Name: proportion, dtype: float64